In [1]:
import os
import sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [17]:
from sparkSession import spark
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [37]:
READ_PATH_CLICKSTREAM = '/Users/ayusman/thisWorks/DE/lakehouse_analytics/lakehouse/bronze/bronze_clickstream'
READ_PATH_USER = '/Users/ayusman/thisWorks/DE/lakehouse_analytics/lakehouse/bronze/bronze_user'

In [38]:
dataframe_clickstream = spark.read.format('delta').load(READ_PATH_CLICKSTREAM)
dataframe_user = spark.read.format('delta').load(READ_PATH_USER)

In [39]:
# uses dictionary comprehension
def trim_everything_lower(dataframe):
    df_trimmed = dataframe.withColumns({c: lower(trim(col(c))) for c in dataframe.columns})
    return df_trimmed

In [21]:
## ensures unique value on passed field list
def ensure_unique(dataframe):
    df_unique = dataframe.dropDuplicates(subset=['event_id'])
    return df_unique

In [22]:
## filter future dates
def filter_future_events(dataframe):
    df_ts = dataframe.withColumn('event_time', upper(trim(col('event_time'))))
    #df_ts = df_ts.withColumn('event_time', to_timestamp(col('event_time'), "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX"))
    df_ts = df_ts.filter(col('event_time') <= current_timestamp())
    return df_ts

In [41]:
## handles null value for the provided columns
## user_id rows, null product_id, quantity rows for event_type purchase, add_to_cart and checkout
def handle_nulls(dataframe):
    # checking null counts accross rows
    dataframe.withColumns({c: sum(col(c).isNull().cast('int')) for c in dataframe_clickstream.columns}).show()
    # replaces with 'unknown' value for columns - category, country, device
    df = dataframe.fillna('unknown', subset=['category', 'country', 'device'])
    # dropping for user_id, event_time, session_id
    df = df.dropna(how='any', subset=['user_id', 'event_time', 'session_id'])
    ## handle NULLs, negatives and 0 in quantity as per event_type
    df = df.filter(~((col('event_type').isin('add_to_cart', 'purchase', 'checkout')) & (col('quantity') <= 0) & (col('quantity').isNull())))
    ## handling null product per event_type
    df = df.filter(~((col('event_type').isin('add_to_cart', 'purchase', 'checkout')) & (col('product_id').isNull())))
    df.withColumns({c: sum(col(c).isNull().cast('int')) for c in dataframe_clickstream.columns}).show()
    # filters negative priced products
    df = df.filter(~((col('price').try_cast('double'))<=0))
    return df

In [75]:
## converts the schema type of all string schema to specfic types
def type_conversion(dataframe):
    schema1 = StructType([
    StructField('category', StringType(), nullable=True),
    StructField('country', StringType(), nullable=True),
    StructField('device', StringType(), nullable=True),
    StructField('event_id', StringType(), nullable=False),
    StructField('event_time', TimestampType(), nullable=False),
    StructField('event_type', StringType(), nullable=True),
    StructField('price', DoubleType(), nullable=True),
    StructField('product_id', StringType(), nullable=True),
    StructField('quantity', IntegerType(), nullable=True),
    StructField('session_id', StringType(), nullable=False),
    StructField('user_id', StringType(), nullable=False),
    StructField('ingestion_ts', TimestampType(), nullable=True),
    StructField('ip_file_name', StringType(), nullable=True),
    ])

    ## nullable = False fields: user_id, event_id, event_time, session_id
    ## this is done to ensure that no nulls value remain in these columns before converting them to nullable=False property
    ## even if there are no nulls, use coalesce anyways
    df = dataframe.withColumns({'user_id': coalesce('user_id', lit('unkonwn')),
                 'event_id': coalesce('event_id', lit('unkonwn')),
                  'event_time': coalesce('event_time', lit('unkonwn')),
                     'session_id': coalesce('session_id', lit('unknown'))
                 })
    df = df.withColumn('event_time', to_timestamp(col('event_time'), "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX"))

    try:
        df = df.to(schema=schema1)
        return df
    except Exception as e:
        print(f'error: {e}')
        

In [73]:

dataframe_clickstream.printSchema()

root
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- ip_file_name: string (nullable = true)



In [74]:
df = trim_everything_lower(dataframe_clickstream)
df = ensure_unique(df)
df = handle_nulls(dataframe_clickstream)
df = filter_future_events(df)
df.printSchema()
df = type_conversion(df)
df.printSchema()

+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
|category|country|device|event_id|event_time|event_type|price|product_id|quantity|session_id|user_id|ingestion_ts|ip_file_name|
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
|      10|      9|    13|       0|         0|         0|    0|        13|     888|         0|      3|           0|           0|
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+

+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
|category|country|device|event_id|event_time|event_type|price|product_id|quantity|session_id|user_id|ingestion_ts|ip_file_name|
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+--

In [ ]:
schema1 = StructType([
    StructField('category', StringType(), nullable=True),
    StructField('country', StringType(), nullable=True),
    StructField('device', StringType(), nullable=True),
    StructField('event_id', StringType(), nullable=False),
    StructField('event_time', TimestampType(), nullable=False),
    StructField('event_type', StringType(), nullable=True),
    StructField('price', DoubleType(), nullable=True),
    StructField('product_id', StringType(), nullable=True),
    StructField('quantity', IntegerType(), nullable=True),
    StructField('session_id', StringType(), nullable=False),
    StructField('user_id', StringType(), nullable=False),
    StructField('ingestion_ts', TimestampType(), nullable=True),
    StructField('ip_file_name', StringType(), nullable=True),
])

In [63]:
## 
df.filter(col('event_time').isNull()).show()
df.withColumns({c: sum(col(c).isNull().cast('int')) for c in dataframe_clickstream.columns}).show()


+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
|category|country|device|event_id|event_time|event_type|price|product_id|quantity|session_id|user_id|ingestion_ts|ip_file_name|
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+

+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
|category|country|device|event_id|event_time|event_type|price|product_id|quantity|session_id|user_id|ingestion_ts|ip_file_name|
+--------+-------+------+--------+----------+----------+-----+----------+--------+----------+-------+------------+------------+
|       0|      0|     0|       0|         0|         0|    0|         9|     873|         0|      0|  

In [50]:
## nullable = False fields: user_id, event_id, event_time 
df = df.withColumns({'user_id': coalesce('user_id', lit('unkonwn')),
                 'event_id': coalesce('event_id', lit('unkonwn')),
                  'event_time': coalesce('event_time', lit('unkonwn')),
                     'session_id': coalesce('session_id', lit('unknown'))
                 })
df = df.withColumn('event_time', to_timestamp(col('event_time'), "yyyy-MM-dd'T'HH:mm:ss.SSSSSSXXX"))
df.printSchema()

root
 |-- category: string (nullable = false)
 |-- country: string (nullable = false)
 |-- device: string (nullable = false)
 |-- event_id: string (nullable = false)
 |-- event_time: timestamp (nullable = false)
 |-- event_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- session_id: string (nullable = false)
 |-- user_id: string (nullable = false)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- ip_file_name: string (nullable = true)



In [51]:
df.to(schema=schema1)

DataFrame[category: string, country: string, device: string, event_id: string, event_time: timestamp, event_type: string, price: double, product_id: string, quantity: int, session_id: string, user_id: string, ingestion_ts: timestamp, ip_file_name: string]

In [52]:
df.printSchema()

root
 |-- category: string (nullable = false)
 |-- country: string (nullable = false)
 |-- device: string (nullable = false)
 |-- event_id: string (nullable = false)
 |-- event_time: timestamp (nullable = false)
 |-- event_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- session_id: string (nullable = false)
 |-- user_id: string (nullable = false)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- ip_file_name: string (nullable = true)

